In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_no_global_pooling import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()

train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 688,354
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling.pth
Epoch   0 | train: 4.9076 | val: 0.3414 | acc: 97.20% | AUC: 0.888  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling.pth
Epoch   1 | train: 0.4476 | val: 0.3144 | acc: 94.55% | AUC: 0.968  | LR: 0.001
Epoch   2 | train: 0.3350 | val: 0.3394 | acc: 93.95% | AUC: 0.967  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling.pth
Epoch   3 | train: 0.2679 | val: 0.1311 | acc: 97.99% | AUC: 0.993  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling.pth
Epoch   4 | train: 0.2098 | val: 0.1080 | acc: 98.39% | AUC: 0.995  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling.pth
Epoch   5 | train: 0.1869 | val: 0.1005 | acc: 98.05% | AUC: 0.996  | LR: 0.001
Epoch   6 | train: 0.1653 | val: 0.0924 | acc: 98.02% | AUC

In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 688,354
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling_fold1.pth
Epoch   0 | train: 6.8446 | val: 0.5303 | acc: 91.41% | AUC: 0.826  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling_fold1.pth
Epoch   1 | train: 0.4574 | val: 0.4024 | acc: 91.69% | AUC: 0.887  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling_fold1.pth
Epoch   2 | train: 0.3486 | val: 0.3340 | acc: 94.52% | AUC: 0.933  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling_fold1.pth
Epoch   3 | train: 0.2925 | val: 0.2459 | acc: 95.49% | AUC: 0.961  | LR: 0.001
Epoch   4 | train: 0.2570 | val: 0.2809 | acc: 90.11% | AUC: 0.959  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_no_global_pooling_fold1.pth
Epoch   5 | train: 0.2359 | val: 0.1976 | 

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.048577,0.117683,97.306085,0.991725,0.976102,0.951852,1838,45,13,257
1,2,0.085932,0.106760,97.493036,0.991525,0.981423,0.929630,1849,35,19,251
2,3,0.076372,0.159213,96.609382,0.985193,0.976645,0.892193,1840,44,29,240
3,4,0.081590,0.127764,97.024640,0.988022,0.976089,0.929368,1837,45,19,250
4,5,0.126718,0.170948,95.264624,0.983804,0.952785,0.951673,1796,89,13,256
5,6,0.112543,0.128261,95.030190,0.991749,0.948514,0.962825,1787,97,10,259
6,7,0.086820,0.161324,97.910864,0.989026,0.989920,0.903346,1866,19,26,243
7,8,0.086092,0.202291,96.607807,0.977683,0.977695,0.884758,1841,42,31,238
8,9,0.102430,0.221335,97.678737,0.978520,0.987798,0.899628,1862,23,27,242
9,10,0.084925,0.161930,97.585887,0.986260,0.987798,0.892193,1862,23,29,240


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result,threshold=0.5,proportion=70)

TypeError: evaluate() got an unexpected keyword argument 'threshold'